# IPI ML — Snowflake access checks

Run top to bottom in a **Snowflake Notebook on Container Runtime** (ideally the GPU runtime, with your artifact repository and any Hugging Face EAI attached in the notebook settings).

Every check is wrapped so a failure prints the error and keeps going. The last cell prints a PASS/FAIL table mapped to the request in the project plan's *IT and access requests* table, so you can paste the failures straight into tickets.

**Key idea:** the artifact repository (`snowflake.snowpark.pypi_shared_repository` or a custom one) serves **Python packages**. Open-weight **model weights** come from somewhere else: Hugging Face through an External Access Integration (EAI), weights uploaded to an internal stage, or the Snowflake Model Registry. Sections 3 and 4 test those separately.

Small credit cost: section 5 makes two tiny Cortex calls, and the notebook itself bills its compute pool while running.

## 0. Settings — edit these first
Placeholders in `ALL_CAPS` strings are guesses; replace them with your real names. Leave a value as `None` to skip that check.

In [ ]:
# --- Project workspace ---
PROJECT_DB      = "YOUR_NONPROD_DB"          # team non-production database
PROJECT_SCHEMA  = "IPI_ML"                   # project schema (requested in week 2)

# --- Source data (fully qualified names) ---
SOURCE_TABLES = {
    "IPI visual return data":   None,  # e.g. "DB.SCHEMA.IPI_VISUAL_RETURNS"
    "SAP asset attributes":     None,
    "Obstruction list":         None,
    "2025 closeout decisions":  None,
}
NOTES_TABLE  = None   # table holding inspector notes, e.g. "DB.SCHEMA.IPI_VISUAL_RETURNS"
NOTES_COLUMN = None   # e.g. "NOTES"

# --- Photos ---
PHOTO_STAGE   = None  # e.g. "@YOUR_NONPROD_DB.IPI_ML.PHOTOS"
SAMPLE_IMAGE  = None  # relative path of one photo in that stage, e.g. "vendorA/04-Photos/123456_1.jpg"

# --- Weights uploaded internally (if HF access is not approved) ---
WEIGHTS_STAGE = None  # e.g. "@YOUR_NONPROD_DB.IPI_ML.MODEL_WEIGHTS"

# --- Cortex models to smoke-test (pick real names from section 5's model list) ---
TEXT_MODEL   = "llama3.1-8b"      # cheap text model
VISION_MODEL = "pixtral-large"    # any vision-capable model enabled for your account

# --- Packages / repo ---
ARTIFACT_REPO = "snowflake.snowpark.pypi_shared_repository"
TRY_RFDETR_WEIGHT_DOWNLOAD = False  # True downloads a ~100+ MB checkpoint to test outbound access

In [ ]:
import sys, os, subprocess, importlib, traceback
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()
results = []   # (area, check, status, detail)

def _record(area, name, ok, detail=""):
    results.append({"area": area, "check": name, "status": "PASS" if ok else "FAIL", "detail": detail[:250]})

def sql_check(area, name, sql, show=True, expect_rows=False):
    """Run SQL; FAIL on error (or on zero rows if expect_rows)."""
    try:
        df = session.sql(sql).to_pandas()
        if show:
            display(df.head(50))
        if expect_rows and df.empty:
            _record(area, name, False, "query ran but returned no rows")
        else:
            _record(area, name, True, f"{len(df)} rows")
        return df
    except Exception as e:
        msg = str(e).strip().splitlines()[0]
        print(f"FAIL [{name}]: {msg}")
        _record(area, name, False, msg)
        return None

def py_check(area, name, fn):
    """Run a Python callable; FAIL on exception. Return value is printed."""
    try:
        out = fn()
        if out is not None:
            print(out)
        _record(area, name, True, "" if out is None else str(out))
        return out
    except Exception as e:
        msg = f"{type(e).__name__}: {str(e).strip().splitlines()[0] if str(e).strip() else ''}"
        print(f"FAIL [{name}]: {msg}")
        _record(area, name, False, msg)
        return None

def skipped(area, name, why):
    results.append({"area": area, "check": name, "status": "SKIP", "detail": why})

## 1. Who and where am I
Confirms the role the notebook runs as. Most "access denied" surprises are the notebook running under a different role than you expect.

In [ ]:
sql_check("Context", "Session context", """
SELECT CURRENT_USER() AS user_name, CURRENT_ROLE() AS role, CURRENT_SECONDARY_ROLES() AS secondary_roles,
       CURRENT_WAREHOUSE() AS warehouse, CURRENT_DATABASE() AS db, CURRENT_SCHEMA() AS schema,
       CURRENT_ACCOUNT() AS account, CURRENT_REGION() AS region, CURRENT_VERSION() AS sf_version
""")
print("Python:", sys.version)
ROLE = session.sql("SELECT CURRENT_ROLE()").collect()[0][0]
PYVER = f"{sys.version_info.major}.{sys.version_info.minor}"

## 2. Compute and GPU
`SHOW COMPUTE POOLS` lists pools your role can see; GPU pools have an `instance_family` starting with `GPU_`. The torch check confirms *this* notebook actually got a GPU.

In [ ]:
pools = sql_check("GPU compute", "Visible compute pools", "SHOW COMPUTE POOLS")
if pools is not None and not pools.empty:
    fam_col = [c for c in pools.columns if "instance_family" in c.lower()]
    if fam_col:
        gpu = pools[pools[fam_col[0]].astype(str).str.upper().str.startswith("GPU")]
        _record("GPU compute", "A GPU pool is visible", not gpu.empty,
                ", ".join(gpu[fam_col[0]].astype(str).unique()) if not gpu.empty else "no GPU_* instance family visible")

def gpu_in_notebook():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("torch sees no CUDA device — notebook is on a CPU runtime/pool")
    p = torch.cuda.get_device_properties(0)
    return f"{torch.cuda.device_count()} GPU(s): {p.name}, {p.total_memory/1e9:.0f} GB VRAM (RF-DETR fine-tuning wants 8 GB+; Qwen3-VL-8B in bf16 wants ~20 GB+)"
py_check("GPU compute", "GPU attached to this notebook", gpu_in_notebook)

## 3. Artifact repository and Python packages
Three layers, from cheapest to most real:
1. **Preinstalled?** What the runtime image already has (`importlib.metadata`).
2. **Resolvable from the repo?** `SYSTEM$RESOLVE_PYTHON_PACKAGES` asks Snowflake to resolve the package against your artifact repository *and* any packages policy (allowlist/blocklist) on the account.
3. **Installable here?** `pip install --dry-run` uses whatever this notebook is actually wired to (selected artifact repo, or a PyPI EAI). If an artifact repo is selected, EAIs are not used for pip.

In [ ]:
sql_check("Packages", "Default artifact repository parameter",
          "SHOW PARAMETERS LIKE 'DEFAULT_PYTHON_ARTIFACT_REPOSITORY' IN ACCOUNT")
sql_check("Packages", "Packages policies visible (allowlist/blocklist)", "SHOW PACKAGES POLICIES IN ACCOUNT")

In [ ]:
# PyPI name -> import name. Grouped by why the project needs it.
PACKAGES = {
    # core / open-weight model runtime
    "torch": "torch", "torchvision": "torchvision", "transformers": "transformers",
    "accelerate": "accelerate", "huggingface-hub": "huggingface_hub", "timm": "timm",
    # detection (1A pole tag)
    "rfdetr": "rfdetr",
    # baselines + tracking
    "autogluon.tabular": "autogluon.tabular", "autogluon.multimodal": "autogluon.multimodal",
    "scikit-learn": "sklearn", "mlflow": "mlflow", "snowflake-ml-python": "snowflake.ml",
    # photo integrity + images
    "imagehash": "imagehash", "pillow": "PIL", "opencv-python-headless": "cv2",
    # PII redaction (or use Cortex AI_REDACT, tested in section 5)
    "presidio-analyzer": "presidio_analyzer", "presidio-anonymizer": "presidio_anonymizer",
}
OPTIONAL = {"vllm": "vllm", "fiftyone": "fiftyone", "label-studio-sdk": "label_studio_sdk"}

from importlib import metadata
rows = []
for pypi, mod in {**PACKAGES, **OPTIONAL}.items():
    try:
        ver = metadata.version(pypi)
    except metadata.PackageNotFoundError:
        ver = None
    rows.append({"package": pypi, "preinstalled_version": ver, "optional": pypi in OPTIONAL})
pre = pd.DataFrame(rows); display(pre)

In [ ]:
# Layer 2: resolve each package against the artifact repo + packages policy (one call per package to isolate failures)
for pypi in PACKAGES:
    q = f"SELECT SYSTEM$RESOLVE_PYTHON_PACKAGES('{PYVER}', $$('{pypi}')$$, '{ARTIFACT_REPO}') AS resolved"
    sql_check("Packages", f"Repo resolves {pypi}", q, show=False)

In [ ]:
# Layer 3: can this notebook actually pip install them? (dry run: nothing is installed)
missing = pre[pre.preinstalled_version.isna() & ~pre.optional].package.tolist()
print("Not preinstalled, testing install path for:", missing)
for pypi in missing:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run", "--quiet", pypi],
                       capture_output=True, text=True, timeout=300)
    ok = r.returncode == 0
    _record("Packages", f"pip can install {pypi}", ok, "" if ok else (r.stderr.strip().splitlines() or ["?"])[-1])
    print(("OK   " if ok else "FAIL ") + pypi)

In [ ]:
# Are the model classes the project needs present in the installed transformers?
# FAIL here usually just means transformers is too old -> pip install -U transformers from the repo.
def has_class(name):
    def _f():
        import transformers
        getattr(transformers, name)
        return f"transformers {transformers.__version__} has {name}"
    return _f
for cls, use in [("Qwen3VLForConditionalGeneration", "open-weight VLM baseline"),
                 ("DINOv3ViTModel", "frozen image embeddings"),
                 ("Siglip2Model", "alt image embeddings"),
                 ("Sam3Model", "SAM 3 pre-labeling")]:
    py_check("Packages", f"transformers class {cls} ({use})", has_class(cls))

## 4. Open-weight model weights
This is the actual "can I get the models" test. Three possible routes; you only need one approved:
- **Hugging Face via EAI** — the notebook must have an EAI attached that allows `huggingface.co` and its CDN hosts. DINOv3 and SAM 3 are **gated**: someone must accept the license on HF and you need a read token (store it as a Snowflake secret and expose it as `HF_TOKEN`).
- **Internal stage** — weights downloaded elsewhere by an approved process and uploaded; check with `LIST`.
- **Model Registry** — models already logged by your team.

In [ ]:
sql_check("Open-weight models", "External access integrations usable by this role", "SHOW EXTERNAL ACCESS INTEGRATIONS")

In [ ]:
HF_REPOS = {
    "Qwen/Qwen3-VL-8B-Instruct":              "open-weight VLM (3D baseline)",
    "facebook/dinov3-vitb16-pretrain-lvd1689m": "DINOv3 embeddings (gated)",
    "google/siglip2-base-patch16-224":        "SigLIP 2 embeddings",
    "facebook/sam3":                          "SAM 3 pre-labeling (gated)",
}
HF_TOKEN = os.environ.get("HF_TOKEN")
print("HF_TOKEN present:", bool(HF_TOKEN))

def hf_probe(repo):
    def _f():
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(repo, "config.json", token=HF_TOKEN)   # tiny file: proves network + license access
        return f"downloaded {os.path.basename(p)}"
    return _f
for repo, use in HF_REPOS.items():
    py_check("Open-weight models", f"HF download {repo} ({use})", hf_probe(repo))

In [ ]:
if TRY_RFDETR_WEIGHT_DOWNLOAD:
    def rfdetr_probe():
        from rfdetr import RFDETRBase
        RFDETRBase()   # pulls the pretrained checkpoint from Roboflow's storage (not Hugging Face)
        return "RF-DETR base checkpoint loaded"
    py_check("Open-weight models", "RF-DETR checkpoint download", rfdetr_probe)
else:
    skipped("Open-weight models", "RF-DETR checkpoint download", "set TRY_RFDETR_WEIGHT_DOWNLOAD=True; needs its own EAI host, not HF")

if WEIGHTS_STAGE:
    sql_check("Open-weight models", "Weights on internal stage", f"LIST {WEIGHTS_STAGE}", expect_rows=True)
else:
    skipped("Open-weight models", "Weights on internal stage", "WEIGHTS_STAGE not set")

def registry_probe():
    from snowflake.ml.registry import Registry
    reg = Registry(session=session, database_name=PROJECT_DB, schema_name=PROJECT_SCHEMA)
    m = reg.show_models()
    display(m)
    return f"{len(m)} models registered in {PROJECT_DB}.{PROJECT_SCHEMA}"
py_check("Experiment tracking", "Model Registry reachable in project schema", registry_probe)

## 5. Cortex AI (hosted models)
- `CORTEX_ENABLED_CROSS_REGION`: `DISABLED` means only models hosted in your account's region. Governance must OK cross-region before you turn it on, since prompts and images leave the region transiently.
- Model access is moving to **model RBAC** (objects in `SNOWFLAKE.MODELS`); the old `CORTEX_MODELS_ALLOWLIST` parameter stops being consulted once the 2026_07 behavior bundle is enabled.
- Images: the stage must use server-side encryption (`SNOWFLAKE_SSE`) and supported formats (.jpg/.png/.webp…), and each model has a per-image size cap.

In [ ]:
sql_check("Cortex", "Cross-region inference setting", "SHOW PARAMETERS LIKE 'CORTEX_ENABLED_CROSS_REGION' IN ACCOUNT")
sql_check("Cortex", "Legacy model allowlist", "SHOW PARAMETERS LIKE 'CORTEX_MODELS_ALLOWLIST' IN ACCOUNT")
sql_check("Cortex", "Models visible to this role (model RBAC)", "SHOW MODELS IN SCHEMA SNOWFLAKE.MODELS", expect_rows=True)

g = sql_check("Cortex", "Grants to current role", f"SHOW GRANTS TO ROLE {ROLE}", show=False)
if g is not None:
    hit = g.apply(lambda r: r.astype(str).str.upper().str.contains("CORTEX_USER").any(), axis=1).any()
    _record("Cortex", "CORTEX_USER database role granted (directly)", bool(hit),
            "" if hit else "not granted directly to this role; may still come via PUBLIC or an inherited role")

In [ ]:
sql_check("Cortex", f"Text call works ({TEXT_MODEL})",
          f"SELECT AI_COMPLETE('{TEXT_MODEL}', 'Reply with the single word OK.') AS reply")

# Redaction for 3A notes — Cortex alternative to Presidio
sql_check("Cortex", "AI_REDACT available (PII in notes)",
          "SELECT AI_REDACT('Tried again 3/14 10am, called Maria at 714-555-0142, left note at 123 Elm St') AS redacted")

In [ ]:
if PHOTO_STAGE and SAMPLE_IMAGE:
    st = PHOTO_STAGE.lstrip("@")
    d = sql_check("Photos", "Photo stage properties (want SNOWFLAKE_SSE + directory enabled)", f"DESCRIBE STAGE {st}")
    sql_check("Photos", "Photo stage has files", f"SELECT COUNT(*) AS n_files, SUM(SIZE)/1e9 AS gb FROM DIRECTORY({PHOTO_STAGE})")
    sql_check("Cortex", f"Vision call works ({VISION_MODEL})", f"""
        SELECT AI_COMPLETE('{VISION_MODEL}',
               'What material is the utility pole in this photo: wood, steel, composite, concrete, or cannot tell? Answer in one word.',
               TO_FILE('{PHOTO_STAGE}', '{SAMPLE_IMAGE}')) AS answer""")
else:
    skipped("Cortex", "Vision call on a real photo", "PHOTO_STAGE / SAMPLE_IMAGE not set")

In [ ]:
# Spend visibility (needs ACCOUNT_USAGE access; FAIL just means ask an admin for a cost report instead)
sql_check("Cortex", "Can see Cortex usage history", """
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_FUNCTIONS_USAGE_HISTORY
ORDER BY START_TIME DESC LIMIT 5""")

## 6. Project schema and privileges
Temporary objects vanish at session end, so these smoke tests leave nothing behind.

In [ ]:
FQ = f"{PROJECT_DB}.{PROJECT_SCHEMA}"
sql_check("Workspace", "Project schema exists", f"SHOW SCHEMAS LIKE '{PROJECT_SCHEMA}' IN DATABASE {PROJECT_DB}", expect_rows=True)
grants = sql_check("Workspace", "Grants on project schema", f"SHOW GRANTS ON SCHEMA {FQ}")
if grants is not None and "privilege" in grants.columns:
    have = set(grants[grants["grantee_name"].str.upper() == ROLE.upper()]["privilege"].str.upper())
    for p in ["USAGE", "CREATE TABLE", "CREATE STAGE", "CREATE VIEW", "CREATE MODEL", "CREATE NOTEBOOK", "CREATE STREAMLIT"]:
        _record("Workspace", f"{p} on schema (direct grant)", p in have or "OWNERSHIP" in have,
                "" if p in have else "not granted directly; may be inherited — confirm with admin if a CREATE fails")

sql_check("Workspace", "Can create a table", f"CREATE TEMPORARY TABLE {FQ}.ZZ_ACCESS_CHECK (x INT)", show=False)
sql_check("Workspace", "Can create a stage", f"CREATE TEMPORARY STAGE {FQ}.ZZ_ACCESS_CHECK_STG ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')", show=False)

## 7. Source data and governance
Read access to every input, and the masking policy on raw notes (the plan says notes stay behind one).

In [ ]:
for label, tbl in SOURCE_TABLES.items():
    if tbl:
        sql_check("Source data", f"Read {label}", f"SELECT COUNT(*) AS n_rows FROM {tbl}")
    else:
        skipped("Source data", f"Read {label}", "table name not set")

if NOTES_TABLE and NOTES_COLUMN:
    db = NOTES_TABLE.split(".")[0]
    pol = sql_check("Governance", "Policies on notes table", f"""
        SELECT policy_name, policy_kind, ref_column_name
        FROM TABLE({db}.INFORMATION_SCHEMA.POLICY_REFERENCES(REF_ENTITY_NAME => '{NOTES_TABLE}', REF_ENTITY_DOMAIN => 'table'))""")
    if pol is not None:
        masked = (pol.get("POLICY_KIND", pd.Series(dtype=str)).astype(str).str.upper().eq("MASKING_POLICY")
                  & pol.get("REF_COLUMN_NAME", pd.Series(dtype=str)).astype(str).str.upper().eq(NOTES_COLUMN.upper())).any()
        _record("Governance", f"Masking policy on {NOTES_COLUMN}", bool(masked), "" if masked else "no masking policy on the notes column yet")
else:
    skipped("Governance", "Masking policy on notes", "NOTES_TABLE / NOTES_COLUMN not set")

## 8. Summary
FAIL/SKIP rows mapped to the matching request in the project plan. Paste the FAIL rows into the ticket.

In [ ]:
REQUEST_FOR_AREA = {
    "Context": "—",
    "GPU compute": "GPU compute (week 6)",
    "Packages": "Python packages (week 3)",
    "Open-weight models": "Open-weight models (week 6)",
    "Experiment tracking": "Experiment tracking (week 6)",
    "Cortex": "Cortex AI (week 4)",
    "Photos": "Openflow / photo folder access / Azure Blob (week 3)",
    "Workspace": "Project schema and role (week 2)",
    "Source data": "Read access to source data (week 2)",
    "Governance": "Use of inspector notes (week 2)",
}
summary = pd.DataFrame(results)
summary["it_request"] = summary["area"].map(REQUEST_FOR_AREA)
pd.set_option("display.max_colwidth", 120)
print(summary["status"].value_counts().to_string(), "\n")
display(summary.sort_values(["status", "area"], key=lambda s: s.map({"FAIL": 0, "SKIP": 1, "PASS": 2}) if s.name == "status" else s))